In [ ]:
%pip install pandas

In [ ]:
import json
import os
import random
from collections import Counter
from pathlib import Path

## Initial Data Exploration

Inspect one model's output file to understand the data structure and category/modifier distribution before processing all files.

In [ ]:
file = 'results/gpt-oss-120b_generated.jsonl'
category_counter = Counter()
modifier_counter = Counter()
with open(file, 'r') as f:
    for line in f:
        data = json.loads(line)
        category_counter[data['category']] += 1
        modifier_counter[data['modifier']] += 1
        
category_counter, modifier_counter, sum(category_counter.values())

In [ ]:
results_folder = 'results/'
result_files = []
for filename in os.listdir(results_folder):
    if filename.endswith('_generated.jsonl'):
        result_files.append(os.path.join(results_folder, filename))
        
len(result_files), result_files

## Completion Coverage Analysis

Count how many times each `(prompt_id, modifier)` pair appears across all model output files. A count of 8 means all 8 models produced a valid compilation for that pair.

In [ ]:
# Count unique codes
unique_codes = set()
for filepath in result_files:
    with open(filepath, 'r') as f:
        for line in f:
            data = json.loads(line)
            unique_codes.add(data['code'])
len(unique_codes)

In [ ]:
completion_counter = Counter()
items_counter = Counter()
for file in result_files:
    with open(file, 'r') as f:
        for line in f:
            data = json.loads(line)
            items_counter[file] += 1
            completion_counter[ (data['prompt_id'], data['modifier']) ] += 1
            
completion_counter, items_counter

In [ ]:
# summarize how many entries have each completion count (e.g., how many 8's, 7's, etc.)
value_hist = Counter(completion_counter.values())

total_keys = len(completion_counter)
total_completions = sum(completion_counter.values())

print("count -> number of (prompt_id, modifier) pairs with that count")
for count in sorted(value_hist.keys(), reverse=True):
    print(f"{count}: {value_hist[count]}")

print(f"\ntotal distinct (prompt_id,modifier) pairs: {total_keys}")
print(f"total completions (sum of all counts): {total_completions}")

## Identify Missing Data

For each `(prompt_id, modifier)` pair, record which model output files are missing a completion. This diagnoses incomplete model runs or prompts that consistently failed to compile.

In [ ]:
# For each result file, collect the set of (prompt_id, modifier) present in that file
presence = {}
for fp in result_files:
    seen = set()
    with open(fp, 'r') as fh:
        for ln in fh:
            obj = json.loads(ln)
            seen.add((obj['prompt_id'], obj['modifier']))
    presence[fp] = seen

# Build a summary for each (prompt_id, modifier) in completion_counter
summary = []
for key in sorted(completion_counter.keys(), key=lambda x: (x[0], x[1])):
    count = completion_counter[key]
    missing = [os.path.basename(fp) for fp, s in presence.items() if key not in s]
    summary.append({
        "prompt_id": key[0],
        "modifier": key[1],
        "completions": count,
        "missing_in_files": missing
    })

# Print a human-readable list and return the structured summary
for item in summary:
    missing_str = ", ".join(item["missing_in_files"]) if item["missing_in_files"] else "(present in all files)"
    print(f"{item['prompt_id']} + {item['modifier']}: {item['completions']} completions; missing in: {missing_str}")

summary

## Select Fully-Covered Pairs

Keep only `(prompt_id, modifier)` pairs where all 8 models produced a valid completion. This ensures every retained sample has a complete multi-model representation.

In [ ]:
# Select all modifier pairs that have exactly 8 completions
most_total_completions = completion_counter.most_common(1)[0][1]
full_completion_pairs = [key for key, count in completion_counter.items() if count == most_total_completions]
full_completion_pairs

In [ ]:
most_common_modifiers = Counter()
for prompt_id, modifier in full_completion_pairs:
    most_common_modifiers[modifier] += 1
most_common_modifiers

## Balance Modifier Distribution

Random-subsample each modifier group down to an equal count so that no modifier dominates the final dataset. Pairs are shuffled before trimming to avoid ordering bias.

In [ ]:
# Balance all modifiers to equal counts by randomly subsampling down to the size of
# the smallest modifier group among fully-covered pairs.
import random
default_pairs = [pair for pair in full_completion_pairs if pair[1] == 'default']
random.shuffle(default_pairs)
parallel_pairs = [pair for pair in full_completion_pairs if pair[1] == 'parallel']
random.shuffle(parallel_pairs)
optimized_pairs = [pair for pair in full_completion_pairs if pair[1] == 'optimized']
random.shuffle(optimized_pairs)
error_pairs = [pair for pair in full_completion_pairs if pair[1] == 'error']
random.shuffle(error_pairs)

default_pairs = default_pairs[:-27]
parallel_pairs = parallel_pairs[:-16]
optimized_pairs = optimized_pairs[:-11]
error_pairs = error_pairs[:-10]
len(default_pairs) + len(parallel_pairs) + len(optimized_pairs) + len(error_pairs)

In [ ]:
balanced_pairs = default_pairs + parallel_pairs + optimized_pairs + error_pairs
balanced_pairs = sorted(balanced_pairs, key=lambda x: (x[0], x[1]))
balanced_pairs[:15]

In [ ]:
with open('results/balanced_full_completion_pairs.json', 'w') as f:
    json.dump(balanced_pairs, f)

## Build Final Dataset

Collect all completions for the selected pairs from every model output file and tag each entry with its source model name.

In [ ]:
# Create combined list with all outputs
results_folder = 'results/'
final_results = []
for filename in os.listdir(results_folder):
    if filename.endswith('_generated.jsonl'):
        with open(os.path.join(results_folder, filename), 'r') as f:
            for line in f:
                data = json.loads(line)
                if (data['prompt_id'], data['modifier']) in balanced_pairs:
                    data['model'] = filename.replace('_generated.jsonl', '')
                    final_results.append(data)
        
len(final_results), final_results

In [ ]:
# Write into jsonl
with open('results/final_completions.jsonl', 'w') as f:
    for item in final_results:
        f.write(json.dumps(item) + '\n')

In [ ]:
# Validate unique codes
unique_codes = set()
for res in final_results:
    unique_codes.add(res['code'])
# Tolerate up to 4 duplicates — in rare cases different models generate identical code
assert len(unique_codes) > len(final_results) - 5, "There are too many duplicates!"

In [ ]:
# Count model distribution
model_counter = Counter()
for res in final_results:
    model_counter[res['model']] += 1
assert len(set(model_counter.values())) == 1, "Model distributions are not balanced!"